In [1]:
import openai
import pandas as pd

from bertopic.representation import OpenAI
from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('hotel_reviews_with_transl.csv', sep = '\t')
df.sample(3)

,id,hotel,review,lang,reviews_transl,reviews_len
5881,6175,Travelodge,A great place to stay I stayed at the Travelod...,en,A great place to stay I stayed at the Travelod...,2156
138,138,Holiday Inn,Would definately be back My wife and I stayed ...,en,Would definately be back My wife and I stayed ...,1169
10272,10806,Park Inn,Petty-minded policy Stayed for four nights in ...,en,Petty-minded policy Stayed for four nights in ...,841


In [3]:
docs = list(df.reviews_transl)

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired

representation_model = KeyBERTInspired()

vectorizer_model = CountVectorizer(min_df=5, stop_words = 'english')
topic_model = BERTopic(nr_topics = 'auto',
                       vectorizer_model = vectorizer_model,
                       representation_model = representation_model)
topics, ini_probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
topic_info = topic_model.get_topic_info()
topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6767,-1_hotel_hotels_rooms_room,"[hotel, hotels, rooms, room, london, inn, hilt...",[Not really value for money! Location is excel...
1,0,3495,0_hotel_hotels_london_paddington,"[hotel, hotels, london, paddington, rooms, hea...",[Great location with wonderful service We book...
2,1,159,1_hotel_rooms_noisy_noise,"[hotel, rooms, noisy, noise, room, quieter, qu...",[Good central London Travelodge We stayed here...
3,2,130,2_hotels_hotel_rooms_room,"[hotels, hotel, rooms, room, cigarette, smoke,...","[A smoking room for a non-smoking family, Good..."
4,3,129,3_bonnington_hotel_hotels_london,"[bonnington, hotel, hotels, london, rooms, tri...",[What have they done to The Bonnington!!! I ha...


In [6]:
topic_info.Representative_Docs

0     [Not really value for money! Location is excel...
1     [Great location with wonderful service We book...
2     [Good central London Travelodge We stayed here...
3     [A smoking room for a non-smoking family, Good...
4     [What have they done to The Bonnington!!! I ha...
5     [Needs a refurbishment! Booked through Travelz...
6     [Good location and less expensive Ouch, London...
7     [Convenient for the O2 arena We stayed here as...
8     [Brilliant hotel! We loved the Millennium Bail...
9     [Very nice 4-star hotel but..., Good 4 star ho...
10    [An affordable modern Park Plaza but a little ...
11    [Pleasantly Suprised and I`m Fussy I Booked th...
12    [Dont book by internet - you may be charged tw...
13    [Excellent service at the Radisson Edwardian B...
14    [Fantastic! We found a bargain deal of 50% off...
15    [RAS except the location, Deceptively Good Loc...
16    [Modern business hotel near Canary Wharf This ...
17    [Well located and comfortable, Well locate

In [7]:
# 方法1: トピックごとに代表ドキュメント1件のみ使用
repr_docs = [docs[0] for docs in topic_info.Representative_Docs]
print(f"代表ドキュメント数: {len(repr_docs)}")

代表ドキュメント数: 46


In [8]:
delimiter = '####'
system_message = "You're a helpful assistant. Your task is to analyse hotel reviews."
user_message = f'''Below is a representative set of customer reviews delimited with {delimiter}. 
Please, identify the main topics mentioned in these comments. 

Return a list of 10-20 topics. 
Output is a JSON list with the following format
[
    {{"topic_name": "<topic1>", "topic_description": "<topic_description1>"}}, 
    {{"topic_name": "<topic2>", "topic_description": "<topic_description2>"}},
    ...
]

Customer reviews:
{delimiter}
{delimiter.join(repr_docs)}
{delimiter}
/no_think'''

messages = [  
    {'role': 'system', 'content': system_message},    
    {'role': 'user', 'content': user_message},  
]

In [9]:
# プロンプトの入力トークン数を確認 (qwen3:1.7b のコンテキスト長は 40,960)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
n_tokens = len(tokenizer.encode(user_message))
print(f"入力トークン数: {n_tokens} / 40960")

入力トークン数: 12040 / 40960


In [10]:
import requests

resp = requests.post(
    'http://host.docker.internal:11434/v1/chat/completions',
    json={
        "model": "qwen3:1.7b",
        "messages": messages,
        "response_format": {"type": "json_object"},
    }
)
print(resp.status_code)

200


In [11]:
resp.json()

{'id': 'chatcmpl-729',
 'object': 'chat.completion',
 'created': 1776147504,
 'model': 'qwen3:1.7b',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '{\n  "summary": "The reviews highlight a mix of experiences across London hotels. Key points include: \\n\\n1. **Holiday Inn Mayfair**: Criticized for bed bugs, poor service, and unpleasant cleanliness. \\n\\n2. **Travelodge London**: Praised for being a good value (£26/night), location, and affordability. \\n\\n3. **Hilton London**: Noted for poor maintenance, unreliable heating, and noise issues. \\n\\n4. **General Trends**: \\n   - Budget-friendly hotels (like Travelodge) are often praised for location and price. \\n   - Mid-range hotels (e.g., Holiday Inn) face criticism about cleanliness and service. \\n   - Some hotels (e.g., Hilton) are highlighted for their basic amenities and location, despite maintenance issues.\\n\\n**Recommendations**: \\n- **Avoid** hotels with 

In [27]:
topic_response = resp.json()['choices'][0]['message']['content']
print(topic_response)

{
  "summary": "The reviews highlight a mix of experiences across London hotels. Key points include: \n\n1. **Holiday Inn Mayfair**: Criticized for bed bugs, poor service, and unpleasant cleanliness. \n\n2. **Travelodge London**: Praised for being a good value (£26/night), location, and affordability. \n\n3. **Hilton London**: Noted for poor maintenance, unreliable heating, and noise issues. \n\n4. **General Trends**: \n   - Budget-friendly hotels (like Travelodge) are often praised for location and price. \n   - Mid-range hotels (e.g., Holiday Inn) face criticism about cleanliness and service. \n   - Some hotels (e.g., Hilton) are highlighted for their basic amenities and location, despite maintenance issues.\n\n**Recommendations**: \n- **Avoid** hotels with known issues (bed bugs, poor service, noise). \n- **Consider** Travelodge for budget-friendly, well-located stays. \n- **Be cautious** with Hilton for maintenance and comfort issues. \n\nThe reviews emphasize that while some hotel

In [13]:
### 方法2: 全代表ドキュメントからランダムサンプリング

import random

# 全トピックの代表ドキュメントをフラットにまとめる
all_repr_docs = topic_info.Representative_Docs.sum()
print(f"全代表ドキュメント数: {len(all_repr_docs)}")

# コンテキスト長に収まるようサンプリング
# プロンプトのテンプレート部分 (~200トークン) + 出力用の余裕 (~2000トークン) を差し引く
max_input_tokens = 40960 - 2200
random.seed(42)
random.shuffle(all_repr_docs)

sampled_docs = []
total_tokens = 0
for doc in all_repr_docs:
    doc_tokens = len(tokenizer.encode(doc))
    if total_tokens + doc_tokens > max_input_tokens:
        break
    sampled_docs.append(doc)
    total_tokens += doc_tokens

print(f"サンプリング後: {len(sampled_docs)} 件, {total_tokens} tokens")

全代表ドキュメント数: 138
サンプリング後: 136 件, 38737 tokens


In [14]:
delimiter = '####'
system_message = "You're a helpful assistant. Your task is to analyse hotel reviews."
user_message_sampled = f'''Below is a representative set of customer reviews delimited with {delimiter}. 
Please, identify the main topics mentioned in these comments. 

Return a list of 10-20 topics. 
Output is a JSON list with the following format
[
    {{"topic_name": "<topic1>", "topic_description": "<topic_description1>"}}, 
    {{"topic_name": "<topic2>", "topic_description": "<topic_description2>"}},
    ...
]

Customer reviews:
{delimiter}
{delimiter.join(sampled_docs)}
{delimiter}
/no_think'''

messages_sampled = [  
    {'role': 'system', 'content': system_message},    
    {'role': 'user', 'content': user_message_sampled},  
]

n_tokens = len(tokenizer.encode(user_message_sampled))
print(f"入力トークン数: {n_tokens} / 40960")

入力トークン数: 38971 / 40960


In [15]:
resp_sampled = requests.post(
    'http://host.docker.internal:11434/v1/chat/completions',
    json={
        "model": "qwen3:1.7b",
        "messages": messages_sampled,
        "response_format": {"type": "json_object"},
    }
)
print(resp_sampled.status_code)

200


In [17]:
resp_sampled.json()

{'id': 'chatcmpl-745',
 'object': 'chat.completion',
 'created': 1776147632,
 'model': 'qwen3:1.7b',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '{"summary": "The hotel reviews highlight a mix of positive and negative experiences, with common issues across different hotels. Key points include: \\n\\n1. **Amenities**: The pool is generally positive, but rooms are described as small and lacking in comfort (e.g., slippers, bathrobes). \\n\\n2. **Service**: Staff are often criticized for being unhelpful, inefficient (e.g., long breakfast queues), and not providing adequate amenities. \\n\\n3. **Room Conditions**: Rooms are described as cramped, unclean, and poorly maintained, with complaints about smells and the lack of basic hotel features. \\n\\n4. **Safety & Noise**: Emergency exits are locked, and noise from sirens is a concern. \\n\\n5. **Location & Value**: Some users felt the hotel didn\'t offer good value for mone

In [29]:
sampled_topic_response = resp_sampled.json()['choices'][0]['message']['content']
print(sampled_topic_response)

{"summary": "The hotel reviews highlight a mix of positive and negative experiences, with common issues across different hotels. Key points include: \n\n1. **Amenities**: The pool is generally positive, but rooms are described as small and lacking in comfort (e.g., slippers, bathrobes). \n\n2. **Service**: Staff are often criticized for being unhelpful, inefficient (e.g., long breakfast queues), and not providing adequate amenities. \n\n3. **Room Conditions**: Rooms are described as cramped, unclean, and poorly maintained, with complaints about smells and the lack of basic hotel features. \n\n4. **Safety & Noise**: Emergency exits are locked, and noise from sirens is a concern. \n\n5. **Location & Value**: Some users felt the hotel didn't offer good value for money, with complaints about pricing and room size. \n\nOverall, while the pool and location are praised, the focus on room size, cleanliness, and service issues dominates the feedback, with specific concerns about emergency exits